# 0. Preliminaries

In [1]:
# this script needs the genre_mapping.csv (from scraping_genre_mapping.py) 
# as well as the artist_genres_raw.csv (from scraping_genre_artists.py) !

# 1. Data Processing

## 1.1 Set path, load raw data

In [2]:
# Import necessary libraries

import sys
import os
from pathlib import Path

# 1) Set project root (parent of this script's directory)
try:
    script_dir = Path(__file__).resolve().parent
except NameError:
    script_dir = Path.cwd()
project_root = script_dir.parent
if not (project_root / "data").is_dir():
    raise RuntimeError(f"Project root {project_root!r} has no data/ folder.")
sys.path.insert(0, str(project_root))

import pandas as pd
import re
import csv

In [3]:
# Load scraped genres
raw_path = project_root / "data" / "scraping" / "genre" / "artist_genres_raw.csv"
df = pd.read_csv(raw_path, dtype=str, keep_default_na=False)

## 1.2 Clean raw data; give each genre its own column entry

In [4]:
# Clean and process genres
# Clean numeric artifacts like [|1|], [|2|], etc.
df['genres_clean'] = df['genres'].str.replace(r'\[\|\d+\|\]', '', regex=True)

# Split on pipe into lists
df['genre_list'] = df['genres_clean'].str.split('|')

# Clean individual genre labels by removing parentheses and stray symbols
def clean_label(label: str) -> str:
    # Remove parenthetical content
    label = re.sub(r'\([^)]*\)', '', label)
    # Keep only letters, digits, spaces, hyphens, and ampersands
    label = re.sub(r'[^A-Za-z0-9\s\-\&]', '', label)
    # Collapse extra spaces and normalize case
    label = re.sub(r'\s+', ' ', label).strip()
    return label.lower()

# Apply cleaning to each entry in genre_list
df['genre_list'] = df['genre_list'].apply(
    lambda lst: [clean_label(g) for g in lst if clean_label(g)]
)

# Determine max genres per artist
max_genres = int(df['genre_list'].apply(lambda lst: len(lst) if isinstance(lst, list) else 0).max())

# Create wide-format columns genre_1 ... genre_N
for idx in range(max_genres):
    col = f'genre_{idx+1}'
    df[col] = df['genre_list'].apply(
        lambda lst: lst[idx].strip().lower() if isinstance(lst, list) and idx < len(lst) else ''
    )
df.drop(columns=['genres', 'genres_clean', 'genre_list'], inplace=True)

In [5]:
# Save cleaned long-format genres

dir = project_root / "data" / "scraping" / "genre"
dir.mkdir(parents=True, exist_ok=True)
path = dir / "artist_genres_long.csv"
df.to_csv(path, index=False, quoting=csv.QUOTE_ALL)
print(f"Saved wide-format genres to {path}")

Saved wide-format genres to /Users/ja_nnik/Project_Music/data/scraping/genre/artist_genres_long.csv


# 1.4 Mapping wiki genres to artists

In [6]:
# Cluster genres into main genres

df_mapping = pd.read_csv(dir / "genre_mapping.csv", dtype=str)
df_artists = pd.read_csv(dir / "artist_genres_long.csv", dtype=str)

# Map subgenres to main genres and error on unmapped
# Build lookup dict: normalized lowercase subgenre -> lowercase main_genre
mapping_dict = {
    row['subgenre'].strip().lower(): row['main_genre'].strip().lower()
    for _, row in df_mapping.iterrows()
    if isinstance(row.get('subgenre'), str) and isinstance(row.get('main_genre'), str)
}

# Identify all genre_x columns
genre_cols = [col for col in df_artists.columns if col.startswith('genre_')]

# Create new main_genre_x columns based on mapping_dict
for col in genre_cols:
    main_col = f"main_{col}"
    df_artists[main_col] = (
        df_artists[col]
          .str.strip()
          .str.lower()
          .map(mapping_dict)
          .fillna('')  # leave empty if no mapping found
    )

# Save clustered genres to CSV
clustered_path = dir / "artist_genres_clustered.csv"
df_artists.to_csv(clustered_path, index=False, quoting=csv.QUOTE_ALL)
print(f"Saved clustered genres to {clustered_path}")

Saved clustered genres to /Users/ja_nnik/Project_Music/data/scraping/genre/artist_genres_clustered.csv


# 2 Unmapped genres

# 2.1 Collect unique OG genre lables

In [7]:
genre_cols = [c for c in df.columns if c.startswith('genre_')]
all_genres = set()
for col in genre_cols:
    all_genres.update(df[col].dropna().unique())

df_unique_genres = pd.DataFrame(sorted(all_genres), columns=['og_genre'])
unique_genres_path = dir / "unique_genres.csv"
df_unique_genres.to_csv(unique_genres_path, index=False)
print(f"Saved {len(df_unique_genres)} unique original genres to {unique_genres_path}")

Saved 1003 unique original genres to /Users/ja_nnik/Project_Music/data/scraping/genre/unique_genres.csv


# 2.2 map each OG genre to main genre from wiki

In [8]:
# Load data for mapping
unique_path  = dir / "unique_genres.csv"
mapping_path = dir / "genre_mapping.csv"

df_unique = pd.read_csv(unique_path, dtype=str).fillna('')

# ─────────────────────────────────────────────────────────────────────────────
#  First pass ─ direct lookup via genre_mapping.csv
# ─────────────────────────────────────────────────────────────────────────────
if "mapping_dict" not in globals():
    df_tmp = pd.read_csv(mapping_path, dtype=str)
    mapping_dict = {
        row["subgenre"].strip().lower(): row["main_genre"].strip().lower()
        for _, row in df_tmp.iterrows()
        if isinstance(row.get("subgenre"), str) and isinstance(row.get("main_genre"), str)
    }

df_unique["mapped_main_genre"] = (
    df_unique["og_genre"].str.strip().str.lower().map(mapping_dict).fillna("")
)

# ─────────────────────────────────────────────────────────────────────────────
#  Second pass ─ word‑fraction fallback (token matching)
# ─────────────────────────────────────────────────────────────────────────────
if (df_unique["mapped_main_genre"] == "").any():
    df_map_full = pd.read_csv(mapping_path, dtype=str).fillna("")

    # Helpers for quick token lookup
    sub_to_main = {
        row["subgenre"].strip().lower(): row["main_genre"].strip().lower()
        for _, row in df_map_full.iterrows()
    }
    main_set = set(df_map_full["main_genre"].str.strip().str.lower())

    # Build a fragment→main_genre lookup for 3+ character words that appear
    # inside any main_genre or subgenre string
    word_to_main = {}
    for mg in main_set:
        for frag in re.split(r"[^a-z0-9]+", mg):
            if len(frag) >= 3:
                word_to_main.setdefault(frag, mg)

    for sub, mg in sub_to_main.items():
        for frag in re.split(r"[^a-z0-9]+", sub):
            if len(frag) >= 3:
                # Don't overwrite an existing mapping that already points to
                # the exact same main genre
                word_to_main.setdefault(frag, mg)

    def token_fallback(row):
        """
        Heuristic mapping for rows still unmapped after the direct look-up.

        Strategy order:
          1. Whole-word search for a main_genre (longest first)
          2. Whole-word search for any subgenre → its main
          3. Token-by-token:
               a. direct match to main_genre
               b. direct match to subgenre
               c. prefix/suffix match (e.g. "afropop" → "pop")
               d. synonym table (e.g. "rap" → "hip hop")
        """
        if row["mapped_main_genre"]:
            return row["mapped_main_genre"]  # already solved

        text = row["og_genre"].strip().lower()

        # Quick pattern checks for composite strings like "r&bsoul" or "rnb"
        if re.search(r"r[\s&]?n?b", text):
            return "r&b & soul"

        # (1) Whole‑word search for main_genre
        for mg in sorted(main_set, key=len, reverse=True):
            if re.search(rf"\b{re.escape(mg)}\b", text):
                return mg

        # (2) Whole‑word search for any known subgenre
        for sub, mg in sub_to_main.items():
            if re.search(rf"\b{re.escape(sub)}\b", text):
                return mg

        # (3) Token-based heuristics ----------------------------------------
        tokens = re.split(r"[\s,\-&]+", text)  # also split on '&'
        for tok in tokens:
            # 3.a direct token == main_genre
            if tok in main_set:
                return tok
            # 3.b direct token == subgenre
            if tok in sub_to_main:
                return sub_to_main[tok]

            # 3.c prefix/suffix contains main_genre (afropop → pop)
            for mg in main_set:
                if tok.endswith(mg) or tok.startswith(mg):
                    return mg

            # 3.d word fragment map (>=3 chars)
            if tok in word_to_main:
                return word_to_main[tok]

            # 3.e synonyms / manual overrides
            SYN_MAP = {
                "rap": "hip hop",
                "hiphop": "hip hop",
                "alt": "",            # ignore noisy prefix
                "alternative": "",     # ignore noisy prefix
                "indie": "",           # ignore noisy prefix
                "urban": "",           # ignore noisy prefix
                "dnb": "electronic",   # catch dnb as electronic
                "edms": "electronic",
                "r&b": "r&b & soul",
                "hawaiian": "world",  # map Hawaiian to world
            }
            if tok in SYN_MAP and SYN_MAP[tok]:
                return SYN_MAP[tok]

        return ""  # still unmapped

    df_unique["mapped_main_genre"] = df_unique.apply(token_fallback, axis=1)

# Save the 2‑column mapping table
mapped_path = dir / "unique_genres_mapped.csv"
df_unique.to_csv(mapped_path, index=False, quoting=csv.QUOTE_ALL)

print(f"Saved explicit og_genre → mapped_main_genre table to {mapped_path}")

 # --- Diagnostics: show unmapped og_genre labels ---------------------------
unmapped = (
    df_unique[df_unique["mapped_main_genre"] == ""]
    .loc[:, "og_genre"]
    .sort_values()
    .unique()
)

print(f"\nUnmapped og_genre count: {len(unmapped)}")
if len(unmapped):
    print("Unmapped labels:")
    for label in unmapped:
        print(f"  • {label}")
else:
    print("Great! All og_genre values are now mapped.")

Saved explicit og_genre → mapped_main_genre table to /Users/ja_nnik/Project_Music/data/scraping/genre/unique_genres_mapped.csv

Unmapped og_genre count: 120
Unmapped labels:
  • 
  • abstract
  • acoustic
  • animation
  • anison
  • aor
  • appalachian
  • australiana
  • author
  • avantgarde
  • ballads
  • bluegrass
  • britfunk
  • broadway
  • brumbeat
  • ccm
  • childrens
  • childrens fiction
  • citation needed
  • countrypopular
  • couplet
  • criticism
  • cueca
  • dansband
  • dansktop
  • darkwave
  • dolewave
  • downtown
  • ebm
  • edm
  • entehno
  • exotica
  • expressionist
  • fantasia
  • fantasy
  • field recordings
  • grebo
  • group sounds
  • hebrew
  • hillbilly
  • humppa
  • idm
  • iskelm
  • jam
  • jingle
  • journalism
  • kaykyoku
  • kayokyoku
  • kirtan
  • lako
  • later years
  • levenslied
  • lyricist
  • maskanda
  • merseybeat
  • microtonal
  • minimalism
  • minimalist
  • mizrahi
  • motown
  • mpb
  • mugham
  • ndw
  • neapolitan
  • no

## 2.3 Apply wiki & "custom" mapping

In [9]:
# Apply mapped genres

# Load long artist‑genre matrix and the explicit mapping table
artist_clean_path = dir / "artist_genres_long.csv"
df_artists_final  = pd.read_csv(artist_clean_path, dtype=str).fillna("")

df_u_map = pd.read_csv(dir / "unique_genres_mapped.csv", dtype=str).fillna("")
u_map_dict = {
    row["og_genre"].strip().lower(): row["mapped_main_genre"].strip().lower()
    for _, row in df_u_map.iterrows()
    if isinstance(row.get("mapped_main_genre"), str) and row["mapped_main_genre"]
}

# Identify original genre_X columns
art_genre_cols = [c for c in df_artists_final.columns if c.startswith("genre_")]

# Create / overwrite main_genre_X columns using the improved mapping
for col in art_genre_cols:
    main_col = f"main_{col}"
    df_artists_final[main_col] = (
        df_artists_final[col]
        .str.strip()
        .str.lower()
        .map(u_map_dict)
        .fillna("")
    )

# ------------------------------------------------------------
# Remove duplicate main‑genre entries per artist
# ------------------------------------------------------------
main_cols = [c for c in df_artists_final.columns if c.startswith("main_genre_")]

for idx, row in df_artists_final.iterrows():
    seen = set()
    for col in main_cols:
        val = row[col]
        if val and val in seen:
            # Duplicate → blank it out
            df_artists_final.at[idx, col] = ""
        else:
            seen.add(val)

# ------------------------------------------------------------
# Ensure every artist has at least one main genre
# ------------------------------------------------------------
zero_mask = (df_artists_final[main_cols] == "").all(axis=1)
if zero_mask.any():
    df_artists_final.loc[zero_mask, "main_genre_1"] = "other"  # Assign "other" to artists with no genres

# Save the fully resolved table
forreal_path = dir / "artist_genres.csv"
df_artists_final.to_csv(forreal_path, index=False, quoting=csv.QUOTE_ALL)
print(f"Saved final artist-genre mapping to {forreal_path}")

Saved final artist-genre mapping to /Users/ja_nnik/Project_Music/data/scraping/genre/artist_genres.csv


# 3. Overview of mapping

In [10]:
# Distribution of main-genre counts

main_cols = [c for c in df_artists_final.columns if c.startswith("main_genre_")]

# Count non-empty main-genre entries per artist
df_artists_final["num_main_genres"] = (df_artists_final[main_cols] != "").sum(axis=1)

dist = df_artists_final["num_main_genres"].value_counts().sort_index()

print("\nDistribution of main-genre counts per artist:")
for k, v in dist.items():
    print(f"  {k} genre(s): {v} artist(s)")

# ------------------------------------------------------------
# List artists that ended up with zero main genres
# ------------------------------------------------------------
zero_main = df_artists_final[df_artists_final["num_main_genres"] == 0]

if len(zero_main):
    print(f"\nArtists with ZERO main_genre entries ({len(zero_main)} total):")
    for _, row in zero_main.iterrows():
        print(f"  •  {row.get('artist_id', '')} – {row.get('common_name', '')}")
else:
    print("\nGreat! Every artist has at least one main genre.")

# --> manual inspection of zero-main artists shows that their wikipedia pages does not list any genres.


Distribution of main-genre counts per artist:
  1 genre(s): 2798 artist(s)
  2 genre(s): 1697 artist(s)
  3 genre(s): 894 artist(s)
  4 genre(s): 357 artist(s)
  5 genre(s): 97 artist(s)
  6 genre(s): 17 artist(s)
  7 genre(s): 7 artist(s)
  8 genre(s): 2 artist(s)
  9 genre(s): 1 artist(s)

Great! Every artist has at least one main genre.


In [11]:
# Frequency table: how often each main genre appears overall

main_genre_counts = (
    df_artists_final[main_cols]
      .replace("", pd.NA)
      .stack()
      .value_counts()
)

print("\nOverall frequency of main genres (across all main_genre_X columns):")
for genre, count in main_genre_counts.items():
    print(f"  {genre}: {count}")


Overall frequency of main genres (across all main_genre_X columns):
  rock: 2529
  pop: 1511
  electronic: 1483
  other: 1436
  jazz: 645
  punk: 556
  blues / country: 530
  metal: 524
  r&b & soul: 464
  world: 403
  folk: 377
  hip hop: 213
  avant-garde & experimental: 157
  classical: 121
  easy listening: 14


# 4. Clean up final csv (deleting genre entries (aka. subgenres / wikipedia genres), renaming main_genre to genre)

In [12]:
# Load the file that still has genre_# + main_genre_# columns
df = pd.read_csv("../data/scraping/genre/artist_genres.csv", dtype=str).fillna("")

# Identify and drop genre_1 … genre_13
genre_cols = [c for c in df.columns if c.startswith("genre_")]
df = df.drop(columns=genre_cols)

# Save the trimmed table
out_path = "../data/scraping/genre/artist_genres.csv"
df.to_csv(out_path, index=False, quoting=csv.QUOTE_ALL)
print(f"Saved genre-trimmed file → {out_path}")

Saved genre-trimmed file → ../data/scraping/genre/artist_genres.csv


In [13]:
# rename main_genre_1 … main_genre_13 to genre_1 … genre_13
rename_map = {f"main_genre_{i}": f"genre_{i}" for i in range(1, 14)}
df = df.rename(columns=rename_map)

# Save the version with the new column names
out_path = "../data/scraping/genre/artist_genres.csv"
df.to_csv(out_path, index=False, quoting=csv.QUOTE_ALL)

print(f"Renamed columns and saved → {out_path}")

Renamed columns and saved → ../data/scraping/genre/artist_genres.csv
